# CaPy Training — Google Colab

**Tri-Modal Contrastive Learning for Drug Discovery**

This notebook trains the CaPy model on a GPU. It is fully self-contained:
install dependencies, download data, preprocess, train, evaluate, and save the checkpoint.

**Data pipeline** (follows Haghighi et al., Nature Methods 2022):
- Downloads the **normalized** CellProfiler file (pre-z-scored by cytominer, all features)
- L1000 expression data is also pre-z-scored — no per-plate norm applied
- Replicate correlation filter (90th percentile quality gate)
- Treatment-level aggregation (compound + dose = one sample)
- L1000 feature detection via `*_at` probe ID pattern
- Scaffold-grouped splitting (all doses of same compound in same split)
- Global RobustScaler (morph) + verify-and-clip (expr)

**Requirements:** Colab Pro runtime with H100 GPU.

## 1. GPU Verification

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime → Change runtime type → GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"Total memory: {gpu_mem:.1f} GB")
print(f"CUDA version: {torch.version.cuda}")
print(f"PyTorch version: {torch.__version__}")

# Enable TF32 for faster float32 ops on H100
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print("TF32 enabled for matmul and cuDNN.")

## 2. Install Dependencies

Colab ships with PyTorch+CUDA pre-installed. We install the remaining packages.

In [ ]:
# Install PyTorch Geometric (must match the Colab PyTorch/CUDA versions)
import torch

TORCH_VERSION = torch.__version__.split('+')[0]  # e.g. '2.6.0'
CUDA_VERSION = torch.version.cuda.replace('.', '')  # e.g. '126'

!pip install -q torch_geometric

# Optional PyG extensions for full functionality.
# If wheels aren't available for this CUDA version, basic GIN training still works.
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION}.html \
    || echo "Optional PyG extensions not available for this CUDA version — proceeding without them."

print("PyG dependencies installed. Remaining deps will be installed via pip install -e . in the next cell.")

## 3. Clone Repository

In [ ]:
import os
import sys
from pathlib import Path

# --- CONFIGURE THIS ---
REPO_URL = "https://github.com/YOUR_USERNAME/capy.git"  # Replace with your repo
REPO_DIR = Path("/content/capy")
# ----------------------

if REPO_DIR.exists():
    # Force-sync to remote — guarantees code matches GitHub even if
    # local files were modified by pip, __pycache__ cleanup, etc.
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main
    print(f"Repository synced to origin/main at {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}
    print(f"Cloned to {REPO_DIR}")

os.chdir(REPO_DIR)

# Flush ALL cached src.* modules and bytecode
for mod_name in [m for m in sys.modules if m.startswith("src")]:
    del sys.modules[mod_name]
!find {REPO_DIR}/src -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null || true

# Upgrade pip and setuptools together to avoid version incompatibility
# (setuptools 78+ requires a compatible pip version)
!pip install -q --upgrade pip setuptools wheel

# Install the package in editable mode so src.* is importable
!pip install -q -e .

# Verify imports work
from src.models.capy import CaPyModel
print("CaPy imports OK")

## 4. Download Rosetta Data

Downloads ~298 MB morphology (normalized CellProfiler, all features, pre-z-scored by cytominer)
and ~25 MB expression (L1000 landmark genes) profiles from Cell Painting Gallery.

In [ ]:
import importlib

import src.data.download
importlib.reload(src.data.download)
from src.data.download import download_rosetta_profiles, download_compound_metadata

RAW_DIR = REPO_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Download morphology (replicate_level_cp_normalized.csv.gz) + expression profiles
profile_paths = download_rosetta_profiles(output_dir=str(RAW_DIR), dry_run=False)
print(f"Profiles downloaded: {profile_paths}")

# Download compound metadata (SMILES, MOA labels)
metadata_path = download_compound_metadata(output_dir=str(RAW_DIR), dry_run=False)
print(f"Metadata downloaded: {metadata_path}")

## 5. Preprocess Data

Runs the pipeline following Haghighi et al.:
1. Load pre-normalized profiles (morphology z-scored by cytominer, L1000 pre-z-scored)
2. Replicate correlation filter (90th percentile)
3. Treatment-level aggregation (compound+dose, mean across replicates)
4. Cross-modal matching by `Metadata_Sample_Dose` / `pert_sample_dose`
5. Feature detection: CellProfiler prefixes for morph, `*_at` suffix for L1000
6. Scaffold-grouped train/val/test split
7. Feature QC (5% NaN threshold, matching reference) + global normalization

In [ ]:
import importlib
import sys

# Purge cached preprocess module RIGHT HERE to guarantee fresh code
for mod_name in [m for m in list(sys.modules) if "preprocess" in m]:
    del sys.modules[mod_name]

import src.data.preprocess
importlib.reload(src.data.preprocess)
from src.data.preprocess import preprocess_pipeline
from src.utils.config import load_config, seed_everything

cfg = load_config(str(REPO_DIR / "configs" / "default.yaml"))
seed_everything(cfg.seed)

# Run full preprocessing pipeline (treatment-level, per-plate norm,
# replicate filter, scaffold split, RobustScaler)
processed_paths = preprocess_pipeline(cfg)
print(f"\nProcessed data saved to: {processed_paths}")

## 6. Weights & Biases Login

Paste your wandb API key when prompted. Get it from https://wandb.ai/authorize

In [ ]:
# Optional: only needed if you set cfg.logging.use_wandb = true in the config
# or override it in cell 14. Skip this cell if you don't use wandb.
import wandb

wandb.login()
print("wandb authenticated.")

## 7. Load Config and Build Data Pipeline

Data is now at the **treatment level** (compound + dose). Multiple treatments
of the same compound share one molecular graph (same SMILES), so we deduplicate
by `compound_id` before featurization to avoid redundant SMILES-to-graph conversion.

In [ ]:
import json

import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.data.dataset import capy_collate_fn, load_split_dataset
from src.data.featurize import featurize_dataset
from src.utils.config import load_config, seed_everything
from src.utils.logging import get_logger, setup_wandb

# Reload config (in case the cell above was re-run)
cfg = load_config(str(REPO_DIR / "configs" / "default.yaml"))

# --- Optional overrides for Colab ---
# cfg.training.batch_size = 512      # H100 80GB can handle larger batches
# cfg.training.epochs = 100          # Shorter run for testing
# cfg.logging.use_wandb = True       # Enable wandb logging
# cfg.training.use_amp = False       # Disable bf16 if you hit issues
# ------------------------------------

seed_everything(cfg.seed)
logger = get_logger("colab")

# Setup wandb
setup_wandb(cfg)

device = "cuda"

# Read feature dimensions from processed data
processed_dir = Path(cfg.data.processed_dir)
with open(processed_dir / "feature_columns.json") as f:
    col_info = json.load(f)
morph_dim = len(col_info["morph_cols"])
expr_dim = len(col_info["expr_cols"])
print(f"Feature dims: morph={morph_dim}, expr={expr_dim}")

# Collect all unique compounds for SMILES featurization.
# Treatment-level data has multiple rows per compound (different doses),
# but each compound has one molecular graph. Deduplicate by compound_id.
all_smiles, all_ids = [], []
seen = set()
for split in ["train", "val", "test"]:
    split_path = processed_dir / f"{split}.parquet"
    if split_path.exists():
        df = pd.read_parquet(split_path)
        for smi, cid in zip(df["smiles"].tolist(), df["compound_id"].tolist()):
            if cid not in seen:
                seen.add(cid)
                all_smiles.append(smi)
                all_ids.append(cid)

print(f"Unique compounds: {len(all_smiles)}")

# Check treatment counts per split
for split in ["train", "val", "test"]:
    split_path = processed_dir / f"{split}.parquet"
    if split_path.exists():
        n = len(pd.read_parquet(split_path))
        print(f"  {split}: {n} treatments")

# Featurize molecules (SMILES -> PyG graphs, keyed by compound_id)
mol_graphs = featurize_dataset(all_smiles, all_ids)
print(f"Featurized {len(mol_graphs)} molecular graphs")

# Build datasets (treatment-level: each row is compound+dose)
train_ds = load_split_dataset(processed_dir, "train", mol_graphs)
val_ds = load_split_dataset(processed_dir, "val", mol_graphs)
test_ds = load_split_dataset(processed_dir, "test", mol_graphs)

print(f"Splits: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

# Validate split sizes
if len(train_ds) == 0:
    raise ValueError(
        "Training dataset is empty — cannot train. "
        "Check preprocessing logs above for dropped compounds."
    )
for name, ds in [("val", val_ds), ("test", test_ds)]:
    if len(ds) == 0:
        print(f"WARNING: {name} dataset is empty — evaluation will be skipped.")

# Build DataLoaders
train_loader = DataLoader(
    train_ds,
    batch_size=cfg.training.batch_size,
    shuffle=True,
    collate_fn=capy_collate_fn,
    drop_last=True,
    num_workers=2,
    persistent_workers=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.training.batch_size,
    shuffle=False,
    collate_fn=capy_collate_fn,
    drop_last=False,
    num_workers=2,
    persistent_workers=True,
)
test_loader = DataLoader(
    test_ds,
    batch_size=cfg.training.batch_size,
    shuffle=False,
    collate_fn=capy_collate_fn,
    drop_last=False,
    num_workers=2,
    persistent_workers=True,
)

print(f"DataLoaders: train={len(train_loader)} batches, "
      f"val={len(val_loader)} batches, test={len(test_loader)} batches")

## 8. Training Run

In [ ]:
from src.models.capy import CaPyModel
from src.training.scheduler import CosineAnnealingWithWarmup
from src.training.trainer import Trainer

# Reset peak memory tracking
torch.cuda.reset_peak_memory_stats()

# Create model
model = CaPyModel(cfg, morph_dim, expr_dim)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

# Optimizer with separate LR for GIN vs MLP
gin_params = list(model.mol_encoder.parameters())
gin_param_ids = {id(p) for p in gin_params}
other_params = [p for p in model.parameters() if id(p) not in gin_param_ids]

optimizer = torch.optim.AdamW(
    [
        {"params": gin_params, "lr": cfg.training.lr_gin},
        {"params": other_params, "lr": cfg.training.lr_mlp},
    ],
    weight_decay=cfg.training.weight_decay,
)

# Scheduler: cosine annealing with warmup
scheduler = CosineAnnealingWithWarmup(
    optimizer, cfg.training.warmup_epochs, cfg.training.epochs
)

# Train
trainer = Trainer(
    cfg, model, train_loader, val_loader, optimizer, scheduler, device
)

print(f"\nStarting training for {cfg.training.epochs} epochs...")
print(f"  Batch size: {cfg.training.batch_size}")
print(f"  LR (MLP): {cfg.training.lr_mlp}, LR (GIN): {cfg.training.lr_gin}")
print(f"  Early stopping: patience={cfg.training.early_stopping_patience}, "
      f"metric={cfg.training.early_stopping_metric}")
print()

best_metrics = trainer.fit()

## 9. Evaluation on Test Set

In [ ]:
from src.evaluation.retrieval import evaluate_all_retrieval

# Load best checkpoint
checkpoint_path = Path(cfg.logging.checkpoint_dir) / "best_model.pt"
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best checkpoint from epoch {checkpoint['epoch']}")
    print(f"Best validation {cfg.training.early_stopping_metric}: "
          f"{checkpoint['best_metric']:.4f}")
else:
    print("No checkpoint found — using model from last training epoch.")

# Embed all test treatments
model.eval()
all_z_mol, all_z_morph, all_z_expr = [], [], []

with torch.no_grad():
    for batch_graphs, morph, expr in test_loader:
        batch_graphs = batch_graphs.to(device)
        morph = morph.to(device)
        expr = expr.to(device)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=getattr(cfg.training, "use_amp", False),
        ):
            outputs = model(batch_graphs, morph, expr)
        all_z_mol.append(outputs["z_mol"].cpu())
        all_z_morph.append(outputs["z_morph"].cpu())
        all_z_expr.append(outputs["z_expr"].cpu())

if not all_z_mol:
    print(f"\nTest loader yielded 0 batches (test_ds has {len(test_ds)} treatments).")
    print("Check preprocessing logs — the test split may be empty or all SMILES "
          "failed to featurize.")
else:
    z_mol = torch.cat(all_z_mol, dim=0).float()
    z_morph = torch.cat(all_z_morph, dim=0).float()
    z_expr = torch.cat(all_z_expr, dim=0).float()

    print(f"\nTest set: {z_mol.shape[0]} treatments, embedding dim={z_mol.shape[1]}")

    # Compute retrieval metrics (includes alignment/uniformity)
    test_metrics = evaluate_all_retrieval(
        z_mol, z_morph, z_expr,
        ks=list(cfg.evaluation.retrieval_ks)
    )

    # Display retrieval results
    print("\n" + "=" * 60)
    print("TEST SET RETRIEVAL METRICS")
    print("=" * 60)

    directions = [
        "mol->morph", "morph->mol",
        "mol->expr", "expr->mol",
        "morph->expr", "expr->morph",
    ]
    for direction in directions:
        metrics_for_dir = {
            k.split("/")[-1]: v
            for k, v in test_metrics.items()
            if k.startswith(direction)
        }
        if metrics_for_dir:
            line = ", ".join(f"{k}={v:.4f}" for k, v in sorted(metrics_for_dir.items()))
            print(f"  {direction:15s}: {line}")

    print("-" * 60)
    mean_metrics = {k: v for k, v in test_metrics.items() if k.startswith("mean_")}
    for k, v in sorted(mean_metrics.items()):
        print(f"  {k:15s}: {v:.4f}")

    # Display alignment & uniformity diagnostics
    print("\n" + "=" * 60)
    print("ALIGNMENT & UNIFORMITY (Wang & Isola 2020)")
    print("=" * 60)
    print("  Alignment (lower = positive pairs closer):")
    for pair in ["mol_morph", "mol_expr", "morph_expr"]:
        print(f"    {pair:15s}: {test_metrics[f'align_{pair}']:.4f}")
    print(f"    {'mean':15s}: {test_metrics['mean_alignment']:.4f}")
    print("  Uniformity (lower = more spread; near 0 = collapsed):")
    for mod in ["mol", "morph", "expr"]:
        print(f"    {mod:15s}: {test_metrics[f'uniform_{mod}']:.4f}")
    print(f"    {'mean':15s}: {test_metrics['mean_uniformity']:.4f}")
    print("=" * 60)

## 9b. t-SNE Visualization of Embedding Space

Visualize the shared 256-dim embedding space projected to 2D. Each treatment
appears as 3 points (one per modality). Well-aligned modalities should cluster
together per treatment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

if "z_mol" in dir():
    # Stack all 3 modalities: [3N, 256]
    all_z = np.vstack([z_mol.numpy(), z_morph.numpy(), z_expr.numpy()])
    n = z_mol.shape[0]
    labels = ["Molecular"] * n + ["Morphology"] * n + ["Expression"] * n

    # t-SNE (fast for ~450 points in 256-dim)
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, n - 1))
    coords = tsne.fit_transform(all_z)

    # Plot
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = {"Molecular": "#e41a1c", "Morphology": "#377eb8", "Expression": "#4daf4a"}
    for modality, color in colors.items():
        mask = np.array(labels) == modality
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            c=color, label=modality, alpha=0.6, s=15, edgecolors="none",
        )
    ax.legend(fontsize=10)
    ax.set_title("t-SNE of CaPy Embedding Space (Test Set)")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping t-SNE — no test embeddings available.")

## 10. Save Checkpoint and Download

In [ ]:
import json as _json

checkpoint_path = Path(cfg.logging.checkpoint_dir) / "best_model.pt"

if checkpoint_path.exists():
    size_mb = checkpoint_path.stat().st_size / 1e6
    print(f"Best checkpoint: {checkpoint_path} ({size_mb:.1f} MB)")

    # Save test metrics alongside checkpoint (if evaluation ran)
    if "test_metrics" in globals():
        results_dir = REPO_DIR / "results"
        results_dir.mkdir(exist_ok=True)
        with open(results_dir / "test_metrics.json", "w") as f:
            _json.dump(test_metrics, f, indent=2)
        print(f"Test metrics saved to {results_dir / 'test_metrics.json'}")
    else:
        print("Test metrics not available (test set may have been empty).")

    # Download to local machine (Colab-specific)
    try:
        from google.colab import files
        files.download(str(checkpoint_path))
        print("\nCheckpoint download started — check your browser downloads.")
    except ImportError:
        print("\nNot running on Colab — checkpoint saved at:", checkpoint_path)
else:
    print("No checkpoint found. Training may not have completed.")

## 11. GPU Memory Usage

In [ ]:
peak_mem = torch.cuda.max_memory_allocated() / 1e9
total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"Peak GPU memory allocated: {peak_mem:.2f} GB")
print(f"Total GPU memory:          {total_mem:.2f} GB")
print(f"Utilization:               {peak_mem / total_mem * 100:.1f}%")

if peak_mem / total_mem > 0.9:
    print("\n⚠ GPU memory nearly full! Consider reducing batch_size.")
elif peak_mem / total_mem < 0.5:
    print("\n💡 Room for a larger batch_size — could improve contrastive learning.")

# Close wandb run cleanly
try:
    wandb.finish()
except Exception:
    pass

print("\nDone! Check your wandb dashboard for training curves.")